In [1]:
import numpy as np

import Util.math_functions as mathf

from Util.Problems import Problem, solution


class P017(Problem):
    number = 17
    title = "Number Letter Counts"
    description = """<p>If the numbers $1$ to $5$ are written out in words: one, two, three, four, five, then there are $3 + 3 + 5 + 4 + 4 = 19$ letters used in total.</p><p>If all the numbers from $1$ to $1000$ (one thousand) inclusive were written out in words, how many letters would be used? </p><br/><p class="note"><b>NOTE:</b> Do not count spaces or hyphens. For example, $342$ (three hundred and forty-two) contains $23$ letters and $115$ (one hundred and fifteen) contains $20$ letters. The use of "and" when writing out numbers is in compliance with British usage.</p>"""
    upper_limit = 1000

In [2]:
p = P017()
p.describe()

## Problem 17: Number Letter Counts

<p>If the numbers $1$ to $5$ are written out in words: one, two, three, four, five, then there are $3 + 3 + 5 + 4 + 4 = 19$ letters used in total.</p><p>If all the numbers from $1$ to $1000$ (one thousand) inclusive were written out in words, how many letters would be used? </p><br/><p class="note"><b>NOTE:</b> Do not count spaces or hyphens. For example, $342$ (three hundred and forty-two) contains $23$ letters and $115$ (one hundred and fifteen) contains $20$ letters. The use of "and" when writing out numbers is in compliance with British usage.</p>

### Solution notes
Sadly, the fastest way to do this is to make it unusable. With some simple tweaks, spaces and hyphens could be added and the numbers_to_words function could produce readable results. But these would only need to be removed again for counting, which is why I leave them out altogether.

In [3]:
@solution(P017, first=True, make_fast=False, warmup_args=(P017.upper_limit, ))
def brute_force(upper_limit):
    def number_to_words(n):
        special_cases = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
        multiples_of_ten = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
        if (n / 10) < 2:
            return special_cases[n]
        number_of_digits = len(str(n))
        if number_of_digits == 4:
            return "onethousand"

        hundreds = ""

        if number_of_digits == 3:
            hundreds_digit = mathf.get_digit(n, 2)
            tens_digit = mathf.get_digit(n, 1)
            ones_digit = mathf.get_digit(n, 0)
            hundreds = f"{special_cases[hundreds_digit]}hundred"
        else:
            tens_digit = mathf.get_digit(n, 1)
            ones_digit = mathf.get_digit(n, 0)

        tens = ""

        if tens_digit == 0 or tens_digit == 1:
            if not (tens_digit == 0 and ones_digit == 0):
                tens = special_cases[tens_digit * 10 + ones_digit]
        else:
            if not ones_digit == 0:
                tens = f"{multiples_of_ten[tens_digit]}{special_cases[ones_digit]}"
            else:
                tens = multiples_of_ten[tens_digit]
        and_string = "and" if hundreds and tens else ""
        return f"{hundreds}{and_string}{tens}"

    total_length = 0
    for i in range(1, upper_limit + 1):
        total_length += len(number_to_words(i))
    return total_length

In [4]:
p.test_all()

21124 found after 1000 tests in 0.683273 ms by brute_force (first)


Now we save all of the names of 2 digits numbers we determine in an array. This way, after 100, we don't need to reason about the 2 digit numbers anymore, we can look them up.

In [5]:
@solution(P017, make_fast=False, warmup_args=(P017.upper_limit, ))
def two_digit_array(upper_limit):
    two_digit_numbers = [""] * 100
    two_digit_numbers[:20] = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
    def number_to_words(n):
        multiples_of_ten = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
        if (n / 10) < 2:
            return two_digit_numbers[n]
        number_of_digits = len(str(n))
        if number_of_digits == 4:
            return "onethousand"

        hundreds = ""

        if number_of_digits == 3:
            hundreds_digit = mathf.get_digit(n, 2)
            tens_digit = mathf.get_digit(n, 1)
            ones_digit = mathf.get_digit(n, 0)
            hundreds = f"{two_digit_numbers[hundreds_digit]}hundred"
        else:
            tens_digit = mathf.get_digit(n, 1)
            ones_digit = mathf.get_digit(n, 0)

        tens = ""
        last_two_digits = tens_digit * 10 + ones_digit
        if last_two_digits != 0:
            if two_digit_numbers[last_two_digits] != "":
                tens = two_digit_numbers[last_two_digits]
            else:
                if not ones_digit == 0:
                    tens = f"{multiples_of_ten[tens_digit]}{two_digit_numbers[ones_digit]}"
                else:
                    tens = multiples_of_ten[tens_digit]
                two_digit_numbers[last_two_digits] = tens
        and_string = "and" if hundreds and tens else ""
        return f"{hundreds}{and_string}{tens}"

    total_length = 0
    for i in range(1, upper_limit + 1):
        total_length += len(number_to_words(i))
    return total_length

In [6]:
p.test_all()

21124 found after 1000 tests in 1.439516 ms by brute_force (first)
21124 found after 1000 tests in 0.595730 ms by two_digit_array


Now to make the function even less usable, but even faster, we separate the calculations of the hundreds. This function could be expanded and made more usable, but for this problem, this is probably as good as I'll get it right now.

In [7]:
@solution(P017, best=True, make_fast=False, warmup_args=(P017.upper_limit, ))
def separated_hundreds(upper_limit):
    hundred_length = len("hundred")
    and_length = len("and")
    upper_limit_digits = len(str(upper_limit))
    max_hundreds = 10
    if upper_limit_digits == 3:
        max_hundreds = mathf.get_digit(upper_limit, 2)
    if upper_limit_digits < 3:
        max_hundreds = 0
    special_cases = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
    def two_digit_numbers_to_words(n):
        multiples_of_ten = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
        if (n / 10) < 2:
            return special_cases[n]

        tens_digit = mathf.get_digit(n, 1)
        ones_digit = mathf.get_digit(n, 0)

        if not ones_digit == 0:
            tens = f"{multiples_of_ten[tens_digit]}{special_cases[ones_digit]}"
        else:
            tens = multiples_of_ten[tens_digit]
        return f"{tens}"

    two_digit_length = 0

    if upper_limit_digits == 2:
        for i in range(1, upper_limit + 1):
            two_digit_length += len(two_digit_numbers_to_words(i))
        return two_digit_length

    for i in range(1, 100):
        two_digit_length += len(two_digit_numbers_to_words(i))

    # Add the length of all 2 digit numbers, as well as that length + the length of "hundred" 100x
    # as well as "and" 99x for all complete hundreds.
    total_length = (two_digit_length + # All 2-digit numbers
                    (two_digit_length + 100 * hundred_length + 99 * and_length) * (max_hundreds - 2))

    # Add whatever is in the last hundred (complete or not)
    upper_limit_ones = mathf.get_digit(upper_limit, 0)
    upper_limit_tens = mathf.get_digit(upper_limit, 1)
    if not (upper_limit_tens == 0 and upper_limit_ones == 0):
        for i in range(upper_limit_tens * 10 + upper_limit_ones):
            if i != 0:
                total_length += and_length + len(two_digit_numbers_to_words(i))
            total_length += (len(special_cases[max_hundreds]) +
                              hundred_length)
    else:
        total_length += (two_digit_length +
                         100 * hundred_length +
                         99 * and_length +
                         100 * len(special_cases[max_hundreds - 1]))
    # Add the prefix for all complete hundreds
    for complete_hundred in range(1, max_hundreds - 1):
        total_length += len(special_cases[complete_hundred]) * 100
    # Only works up to 1000
    if upper_limit_digits == 4:
        total_length += len("onethousand")
    return total_length

In [8]:
p.test_all()

21124 found after 1000 tests in 0.595700 ms by brute_force (first)
21124 found after 1000 tests in 0.027460 ms by separated_hundreds (best)
21124 found after 1000 tests in 0.549630 ms by two_digit_array
